In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# ==================================================
# 1. CONFIGURATION & PATHS
# ==================================================

# UPDATE THIS PATH to match your Kaggle Input directory!
DATA_DIR = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset/' 
METADATA_PATH = os.path.join(DATA_DIR, 'metadata.csv')

# ==================================================
# 2. EXTRACT RAW DATA
# ==================================================
print("Starting Data Extraction...")
metadata = pd.read_csv(METADATA_PATH)

print(metadata.head())
print(metadata.columns)

# We only want discharge cycles for our 4 specific batteries
target_batteries = ['B0005', 'B0006', 'B0007', 'B0018']
discharge_meta = metadata[(metadata['type'] == 'discharge') & 
                          (metadata['battery_id'].isin(target_batteries))]

extracted_data = []

for _, row in discharge_meta.iterrows():
    batt_id = row['battery_id']
    capacity = row['Capacity']
    filename = row['filename']
    
    # Assuming raw files are stored in a 'raw_data' subfolder. Update if different.
    file_path = os.path.join(DATA_DIR, 'data', filename) 
    
    try:
        raw_df = pd.read_csv(file_path)
        mean_v = raw_df['Voltage_measured'].mean()
        std_v = raw_df['Voltage_measured'].std()
        mean_c = raw_df['Current_measured'].mean()
        max_t = raw_df['Temperature_measured'].max()
        time_sec = raw_df['Time'].max()
        
        extracted_data.append([batt_id, mean_v, std_v, mean_c, max_t, time_sec, capacity])
    except Exception as e:
        print(f"Could not read {filename}: {e}")

cols = ['battery_id', 'mean_voltage', 'std_voltage', 'mean_current', 'max_temperature', 'discharge_time_sec', 'capacity_Ah']
df = pd.DataFrame(extracted_data, columns=cols)

# 1. Force the capacity column to be numeric (turns bad text into NaN)
df['capacity_Ah'] = pd.to_numeric(df['capacity_Ah'], errors='coerce')

# 2. Drop any corrupted rows that became NaN
df = df.dropna(subset=['capacity_Ah']).reset_index(drop=True)

# 3. Correct SOH calculation based on official 2.0 Ahr rating
df['SOH'] = df['capacity_Ah'] / 2.0

# ==================================================
# 3. CHRONOLOGICAL SCALING
# ==================================================
print("Scaling Features...")
features_to_scale = ['mean_voltage', 'std_voltage', 'mean_current', 'max_temperature', 'discharge_time_sec', 'capacity_Ah']

# Fit scaler ONLY on the training batteries
train_mask = df['battery_id'].isin(['B0005', 'B0006', 'B0007'])
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, features_to_scale])

df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.transform(df[features_to_scale])

# ==================================================
# 4. SLIDING WINDOW CREATION (PER BATTERY)
# ==================================================
print("Creating Sliding Windows...")
WINDOW_SIZE = 10
X_list, y_list, batt_list = [], [], []

# Group by battery ID so windows don't cross between different batteries
for batt_id, group in df_scaled.groupby('battery_id'):
    group = group.reset_index(drop=True)
    if len(group) < WINDOW_SIZE:
        continue
        
    for i in range(len(group) - WINDOW_SIZE + 1):
        window = group.iloc[i:i+WINDOW_SIZE]
        X_seq = window[features_to_scale].values.flatten()
        y_val = window['SOH'].iloc[-1]
        
        X_list.append(X_seq)
        y_list.append(y_val)
        batt_list.append(batt_id)

feature_names = [f"{col}_t{-WINDOW_SIZE + i + 1}" for i in range(WINDOW_SIZE) for col in features_to_scale]

final_df = pd.DataFrame(X_list, columns=feature_names)
final_df['SOH_target'] = y_list
final_df['battery_id'] = batt_list

# ==================================================
# 5. TRUE CELL-LEVEL SPLIT
# ==================================================
print("Splitting Dataset...")

# Train: B0005, B0006, B0007
train_df = final_df[final_df['battery_id'].isin(['B0005', 'B0006', 'B0007'])].drop(columns=['battery_id'])

# Validation and Test: B0018
b0018_df = final_df[final_df['battery_id'] == 'B0018'].drop(columns=['battery_id']).reset_index(drop=True)

# Splits B0018 sequentially as documented in Phase 1
val_df = b0018_df.iloc[:50]
test_df = b0018_df.iloc[50:]

print(f"✓ Train Samples: {len(train_df)}")
print(f"✓ Val Samples: {len(val_df)}")
print(f"✓ Test Samples: {len(test_df)}")

# ==================================================
# 6. EXPORT ARTIFACTS
# ==================================================
train_df.to_csv('nasa_train.csv', index=False)
val_df.to_csv('nasa_val.csv', index=False)
test_df.to_csv('nasa_test.csv', index=False)

with open('nasa_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Preprocessing Complete! All files saved to Kaggle output (/kaggle/working/).")